# MRO Parts Catalog — Full Cleaning Pipeline
**Goal:** Produce a clean, normalized dataset ready for the AI Part Interchange Recommender.

### Pipeline stages
1. Load & baseline audit
2. Drop useless columns
3. Build a consolidated core record per part
4. Normalize manufacturer names & part numbers
5. Strip HTML from description fields
6. Parse & normalize the `attribute_table` column
7. Price & weight sanity checks
8. Deduplication
9. Export clean dataset

---
## Stage 0 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import ast
from html.parser import HTMLParser

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

RAW_PATH   = 'data/data.csv'
CLEAN_PATH = 'data/cleaned.csv'

print('Imports OK')

---
## Stage 1 — Load & Baseline Audit

In [ ]:
data = pd.read_csv(RAW_PATH, dtype={28: 'string', 66: 'string'}, low_memory=False)
data.columns = data.columns.str.lower().str.strip()

print(f'Raw shape: {data.shape[0]:,} rows × {data.shape[1]} columns')

In [ ]:
# Full null audit — save so you can refer back to it
null_audit = (
    pd.DataFrame({
        'pct_null':   (data.isna().mean() * 100).round(2),
        'null_count': data.isna().sum(),
        'dtype':      data.dtypes.astype(str),
        'n_unique':   data.nunique(),
    })
    .sort_values('pct_null', ascending=False)
    .reset_index()
    .rename(columns={'index': 'column'})
)

null_audit.to_csv('data/null_audit.csv', index=False)
print('Null audit saved → data/null_audit.csv')
null_audit.head(20)

---
## Stage 2 — Drop Useless Columns

Three categories of columns to drop:
- **100% null** — no information at all
- **Single-value / constant** — no variance, useless for matching
- **Operational / non-semantic** — inventory flags, CMS slugs, meta keywords that don't describe the part itself

In [ ]:
# --- 2a. Drop 100% null columns ---
cols_all_null = null_audit.loc[null_audit['pct_null'] == 100, 'column'].tolist()
data.drop(columns=cols_all_null, inplace=True)
print(f'Dropped {len(cols_all_null)} fully-null columns. Shape: {data.shape}')

In [ ]:
# --- 2b. Drop constant columns (only 1 unique non-null value) ---
constant_cols = [c for c in data.columns if data[c].nunique(dropna=True) <= 1]
data.drop(columns=constant_cols, inplace=True)
print(f'Dropped {len(constant_cols)} constant columns: {constant_cols}')
print(f'Shape: {data.shape}')

In [ ]:
# --- 2c. Drop operational / CMS / inventory columns ---
# These columns carry no part-identity or specs signal useful for interchange matching.
# Adjust this list if any turn out to be needed later.
operational_cols = [
    # Inventory management flags
    'backorder.value', 'manageinventory.value', 'maximumquantitytoorder.value',
    'lowinventorythreshold.value', 'highlightlowinventory.value',
    'inventorythreshold.value', 'minimumquantitytoorder.value',
    'decrementquantity.value', 'isupcoming.value', 'availability_date',
    'custom_quantity_increment',
    # CMS / slug / meta — SEO fields, not part specs
    'slugprototypes.livhaven.value', 'slugprototypes.mro.fallback',
    'metakeywords.default.value', 'metakeywords.livhaven.fallback',
    'metakeywords.mro.fallback',
    'metatitles.mro.value', 'metatitles.mro.fallback',
    'metatitles.livhaven.value', 'metatitles.livhaven.fallback',
    'metadescriptions.mro.value', 'metadescriptions.mro.fallback',
    'metadescriptions.livhaven.value', 'metadescriptions.livhaven.fallback',
    # UPC rarely populated, not needed for interchange
    'upc',
]

# Only drop columns that still exist (some may have been removed already)
to_drop = [c for c in operational_cols if c in data.columns]
data.drop(columns=to_drop, inplace=True)
print(f'Dropped {len(to_drop)} operational columns. Shape: {data.shape}')

---
## Stage 3 — Build a Consolidated Core Record

The dataset has many *fallback* and *variant* description/name columns across storefronts 
(`.mro.value`, `.livhaven.value`, `.default.value`, etc.). We collapse each logical field 
into one column by taking the first non-null value in a priority order.

In [ ]:
def coalesce(*series):
    """Return first non-null value across a list of Series, element-wise."""
    result = series[0].copy()
    for s in series[1:]:
        if s.name in data.columns:
            result = result.combine_first(s)
    return result

def col(name):
    """Safely fetch a column or return a null Series."""
    return data[name] if name in data.columns else pd.Series([np.nan]*len(data), index=data.index)

# Consolidated name: prefer MRO → LivHaven → default
data['name_clean'] = coalesce(
    col('names.mro.fallback'),
    col('names.livhaven.value'),
    col('names.livhaven.fallback'),
)

# Consolidated short description
data['short_desc_clean'] = coalesce(
    col('shortdescriptions.mro.fallback'),
    col('shortdescriptions.livhaven.value'),
    col('shortdescriptions.livhaven.fallback'),
    col('shortdescriptions.mro.value'),
)

# Consolidated long description
data['description_clean'] = coalesce(
    col('descriptions.mro.value'),
    col('descriptions.mro.fallback'),
    col('descriptions.livhaven.value'),
    col('descriptions.livhaven.fallback'),
    col('descriptions.default.value'),
    col('manufacturer_description'),
)

# Drop the now-redundant storefront-specific columns
redundant = [
    c for c in data.columns
    if any(c.startswith(p) for p in [
        'names.', 'shortdescriptions.', 'descriptions.',
        'metatitles.', 'slugprototypes.', 'metadescriptions.'
    ])
]
data.drop(columns=redundant, inplace=True)

print(f'Core record columns built. Shape: {data.shape}')
print('\nNew consolidated columns:')
print(data[['name_clean', 'short_desc_clean', 'description_clean']].notnull().mean().mul(100).round(1))

---
## Stage 4 — Normalize Manufacturer Names & Part Numbers

This is critical for the recommender — matching fails completely if `"Parker"` and `"parker "` 
are treated as different manufacturers, or if part numbers have inconsistent spacing/casing.

In [ ]:
# --- 4a. Identify your manufacturer column ---
# Based on the R analysis this is likely 'brand_default_title'. Adjust if needed.
MFR_COL  = 'brand_default_title'   # <-- change if your column name differs
PART_COL = 'manufacturer_part_number'

print('Manufacturer column sample (raw):')
print(data[MFR_COL].value_counts().head(20))

In [ ]:
# --- 4b. Normalize manufacturer names ---

# Known aliases — expand this dict as you discover more
MFR_ALIASES = {
    'bosch rexroth': 'Bosch Rexroth',
    'rexroth': 'Bosch Rexroth',
    'parker hannifin': 'Parker',
    'parker-hannifin': 'Parker',
    'smc corporation': 'SMC',
    'smc corp': 'SMC',
    'aventics': 'Aventics',
    'versa': 'Versa',
    # Add more as you discover them
}

def normalize_manufacturer(name):
    if pd.isna(name):
        return np.nan
    cleaned = str(name).strip()
    lookup  = cleaned.lower()
    return MFR_ALIASES.get(lookup, cleaned)  # fall back to title-stripped original

data['manufacturer_norm'] = data[MFR_COL].apply(normalize_manufacturer)

print('Manufacturer name coverage after normalization:')
print(f"  Non-null: {data['manufacturer_norm'].notna().sum():,}")
print(f"  Unique:   {data['manufacturer_norm'].nunique():,}")
print(data['manufacturer_norm'].value_counts().head(15))

In [ ]:
# --- 4c. Normalize part numbers ---

def normalize_part_number(pn):
    """
    Standardize part numbers:
    - Strip whitespace
    - Uppercase
    - Collapse internal whitespace to single space
    - Remove non-alphanumeric except hyphens (adjust if your parts use other chars)
    """
    if pd.isna(pn):
        return np.nan
    pn = str(pn).strip().upper()
    pn = re.sub(r'\s+', ' ', pn)          # collapse whitespace
    pn = re.sub(r'[^A-Z0-9\-/ ]', '', pn) # keep alphanumeric, dash, slash, space
    return pn.strip()

data['part_number_norm'] = data[PART_COL].apply(normalize_part_number)

# Sanity check: how many are null?
print(f'Part number null after normalization: {data["part_number_norm"].isna().sum():,}')
print('\nSample normalized part numbers:')
print(data[[PART_COL, 'part_number_norm']].dropna().head(10))

In [ ]:
# --- 4d. Flag suspiciously short or long part numbers ---
data['part_number_length'] = data['part_number_norm'].str.len()

print('Part number length distribution:')
print(data['part_number_length'].describe())

# Per the project doc, part numbers are 9-10 digits — flag outliers
outlier_pns = data[data['part_number_length'].notna() & 
                   ((data['part_number_length'] < 4) | (data['part_number_length'] > 30))]
print(f'\nPart numbers with suspicious length (<4 or >30): {len(outlier_pns):,}')
print(outlier_pns[[PART_COL, 'part_number_norm', 'part_number_length']].head(10))

---
## Stage 5 — Strip HTML From Description Fields

In [ ]:
class _HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.reset()
        self.fed = []
    def handle_data(self, d):
        self.fed.append(d)
    def get_text(self):
        return ' '.join(self.fed)

def strip_html(text):
    if pd.isna(text):
        return np.nan
    text = str(text)
    if '<' not in text:          # fast path — no HTML
        return text.strip()
    s = _HTMLStripper()
    s.feed(text)
    return re.sub(r'\s+', ' ', s.get_text()).strip()

for col_name in ['description_clean', 'short_desc_clean', 'name_clean']:
    if col_name in data.columns:
        data[col_name] = data[col_name].apply(strip_html)

# Also strip manufacturer_description if it still exists
if 'manufacturer_description' in data.columns:
    data['manufacturer_description'] = data['manufacturer_description'].apply(strip_html)

print('HTML stripped from description fields.')
print('\nSample cleaned description:')
print(data['description_clean'].dropna().iloc[0])

---
## Stage 6 — Parse the `attribute_table` Column

This column (~60% populated) likely holds structured specs (pressure, dimensions, flow rate etc.) 
as a serialized dict or JSON string. We'll parse it into a flat dataframe and merge the most 
important attributes back as their own columns.

In [ ]:
# First, inspect what the raw values look like
attr_sample = data['attribute_table'].dropna().head(5)
for i, v in enumerate(attr_sample):
    print(f'--- Sample {i+1} ---')
    print(repr(v[:300]))
    print()

In [ ]:
def parse_attribute_table(raw):
    """
    Try to parse the attribute_table cell into a flat dict.
    Handles JSON strings, Python dict strings, and list-of-dicts.
    Returns {} on failure.
    """
    if pd.isna(raw) or str(raw).strip() in ('', 'nan'):
        return {}
    raw = str(raw).strip()
    # Try JSON first
    try:
        parsed = json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        # Fall back to Python literal eval
        try:
            parsed = ast.literal_eval(raw)
        except Exception:
            return {'_parse_error': raw[:100]}

    # If it's a list of {label, value} dicts, flatten it
    if isinstance(parsed, list):
        flat = {}
        for item in parsed:
            if isinstance(item, dict):
                key = item.get('label') or item.get('name') or item.get('key')
                val = item.get('value') or item.get('val')
                if key:
                    flat[str(key).strip()] = str(val).strip() if val is not None else np.nan
        return flat
    if isinstance(parsed, dict):
        return {str(k).strip(): str(v).strip() for k, v in parsed.items()}
    return {}

print('Parsing attribute_table... (may take ~30 seconds on 300k rows)')
parsed_attrs = data['attribute_table'].apply(parse_attribute_table)
attr_df = pd.DataFrame(parsed_attrs.tolist(), index=data.index)

print(f'Attribute columns extracted: {len(attr_df.columns)}')
print('Top 20 most populated attribute keys:')
print(attr_df.notna().sum().sort_values(ascending=False).head(20))

In [ ]:
# Keep only attributes that appear in at least 1% of rows (tune as needed)
MIN_COVERAGE = 0.01
keep_attrs = attr_df.columns[
    attr_df.notna().mean() >= MIN_COVERAGE
].tolist()

# Prefix to avoid column name collisions
attr_df_filtered = attr_df[keep_attrs].add_prefix('attr_')

# Merge back into main dataframe
data = pd.concat([data, attr_df_filtered], axis=1)

print(f'Merged {len(keep_attrs)} attribute columns back into main dataset.')
print(f'Shape: {data.shape}')

---
## Stage 7 — Price & Weight Sanity Checks

In [ ]:
# Identify numeric price/weight columns — adjust names to match your dataset
PRICE_COL  = 'last_sold_price'  # from R analysis
WEIGHT_COL = 'item_weight'

for col_name in [PRICE_COL, WEIGHT_COL]:
    if col_name not in data.columns:
        print(f'Column {col_name!r} not found — skipping')
        continue

    series = pd.to_numeric(data[col_name], errors='coerce')
    data[col_name] = series  # coerce any strings to NaN

    zero_pct = (series == 0).mean() * 100
    neg_pct  = (series < 0).mean() * 100
    p99      = series.quantile(0.99)

    print(f'\n{col_name}:')
    print(f'  null:    {series.isna().sum():,} ({series.isna().mean()*100:.1f}%)')
    print(f'  zero:    {(series==0).sum():,} ({zero_pct:.1f}%)')
    print(f'  negative:{(series<0).sum():,} ({neg_pct:.1f}%)')
    print(f'  p99:     {p99:.2f}')
    print(f'  max:     {series.max():.2f}')

    # Flag zeros as NaN — $0 price is almost always missing/bad data
    if col_name == PRICE_COL:
        data[col_name] = series.replace(0, np.nan)
        print(f'  → Zeros replaced with NaN')

---
## Stage 8 — Deduplication

Parts may appear multiple times across storefronts. 
We deduplicate on `(manufacturer_norm, part_number_norm)` — the primary key for interchange matching.

In [ ]:
dupe_key = ['manufacturer_norm', 'part_number_norm']

n_total = len(data)
n_unique = data.drop_duplicates(subset=dupe_key).shape[0]
n_dupes = n_total - n_unique

print(f'Total rows:       {n_total:,}')
print(f'Unique (mfr+pn):  {n_unique:,}')
print(f'Duplicate rows:   {n_dupes:,} ({n_dupes/n_total*100:.1f}%)')

# Show the worst offenders
top_dupes = (
    data.groupby(dupe_key).size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(10)
)
print('\nMost duplicated part+manufacturer combos:')
print(top_dupes)

In [ ]:
# De-duplicate strategy:
# For duplicates, keep the row with the most non-null values (richest record)

data['_non_null_count'] = data.notna().sum(axis=1)

data_deduped = (
    data
    .sort_values('_non_null_count', ascending=False)  # richest first
    .drop_duplicates(subset=dupe_key, keep='first')
    .drop(columns=['_non_null_count'])
    .reset_index(drop=True)
)

print(f'Shape after deduplication: {data_deduped.shape}')

---
## Stage 9 — Final Column Review & Export

In [ ]:
# Final null rates across surviving columns
final_audit = (
    pd.DataFrame({
        'pct_null':   (data_deduped.isna().mean() * 100).round(2),
        'null_count': data_deduped.isna().sum(),
        'n_unique':   data_deduped.nunique(),
    })
    .sort_values('pct_null', ascending=False)
    .reset_index()
    .rename(columns={'index': 'column'})
)

print(f'Final shape: {data_deduped.shape}')
print()
print('Core recommender columns — coverage:')
core_cols = [
    'manufacturer_norm', 'part_number_norm',
    'name_clean', 'short_desc_clean', 'description_clean',
    PRICE_COL, WEIGHT_COL
]
for c in core_cols:
    if c in data_deduped.columns:
        pct_filled = data_deduped[c].notna().mean() * 100
        print(f'  {c:<30} {pct_filled:5.1f}% filled')

print()
print(f'Attribute columns extracted: {len([c for c in data_deduped.columns if c.startswith("attr_")])}')
print('\nAll columns:')
print(list(data_deduped.columns))

In [ ]:
data_deduped.to_csv(CLEAN_PATH, index=False)
print(f'✅ Clean dataset saved → {CLEAN_PATH}')
print(f'   Rows:    {len(data_deduped):,}')
print(f'   Columns: {len(data_deduped.columns)}')

---
## What's Next

Once this pipeline runs cleanly, the next step toward the recommender is:

1. **Text embedding** — Embed `name_clean + short_desc_clean + attr_*` fields using a sentence 
   transformer (e.g., `all-MiniLM-L6-v2`) to build a semantic similarity index.
   
2. **Attribute matching** — For parts in the same category, compare structured `attr_*` values 
   (pressure, dimensions, port size) to produce a confidence score.

3. **Candidate retrieval** — For a given `(manufacturer, part_number)` query, retrieve the 
   top-K nearest neighbors by embedding distance, then re-rank by attribute overlap.

4. **Confidence tiering** — Map final score to Green / Yellow / Red:
   - 🟢 Green: High embedding similarity + strong attribute match
   - 🟡 Yellow: Good embedding similarity, partial attribute match
   - 🔴 Red: Weak signal, requires human review
